# RESEARCH NOTEBOOK --> SmugPlug

In [1]:
import os
import sys
from decimal import Decimal
import warnings
import asyncio
import aiohttp
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime
import time
from typing import Dict, Optional, Tuple

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [2]:
import pandas as pd
import pandas_ta as ta  # noqa: F401
from core.data_sources import CLOBDataSource

# Initialize the data source
clob = CLOBDataSource()

In [3]:
# Define the parameters
exchange = "binance_perpetual"
trading_pair = "WLD-USDT"
timeframe = "1h"
days = 30

In [4]:
# Get the candles
candles = await clob.get_candles_last_days(
    exchange, trading_pair, timeframe, days, from_trades=False)

2025-02-05 23:17:01,296 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16a45da80>
2025-02-05 23:17:01,297 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x103e8a380>, 517653.045373333)])']
connector: <aiohttp.connector.TCPConnector object at 0x16a45dab0>


In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [6]:
async def get_orderbook_data():
    connector = None
    try:
        # Get Binance order book through connector
        connector = clob.get_connector("binance_perpetual")
        orderbook = await connector._orderbook_ds._order_book_snapshot(trading_pair)
        
        # Create lists to store the data
        data = []
        
        # Add bids
        for price, amount, _ in orderbook.bids[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'bid'
            })
            
        # Add asks
        for price, amount, _ in orderbook.asks[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'ask'
            })
        
        # Create DataFrame from the list of dictionaries
        orderbook_df = pd.DataFrame(data)
        
        # Calculate cumulative amounts
        orderbook_df['cumulative_amount'] = orderbook_df.groupby('side')['amount'].cumsum()
        
        return orderbook_df
        
    except Exception as e:
        print(f"Error fetching order book data: {str(e)}")
        return None
    finally:
        # Properly close the connector and session
        if connector and hasattr(connector, '_session') and not connector._session.closed:
            await connector._session.close()
        if hasattr(clob, '_session') and not clob._session.closed:
            await clob._session.close()

# Get and display the order book data
orderbook_df = await get_orderbook_data()

if orderbook_df is not None:
    print("\nOrder Book Data:")
    print(orderbook_df)
    
    # You can also create a more detailed view of market depth
    print("\nMarket Depth Analysis:")
    print("\nBids Summary:")
    bids = orderbook_df[orderbook_df['side'] == 'bid'].head()
    print(bids)
    
    print("\nAsks Summary:")
    asks = orderbook_df[orderbook_df['side'] == 'ask'].head()
    print(asks)
    
    # Calculate and print some market metrics
    best_bid = bids['price'].iloc[0]
    best_ask = asks['price'].iloc[0]
    spread = best_ask - best_bid
    spread_pct = (spread / best_bid) * 100
    
    print(f"\nMarket Metrics:")
    print(f"Best Bid: {best_bid:.8f}")
    print(f"Best Ask: {best_ask:.8f}")
    print(f"Spread: {spread:.8f} ({spread_pct:.4f}%)")
    
    # Create a visualization of the order book
    fig = go.Figure()
    
    # Add bids
    bids = orderbook_df[orderbook_df['side'] == 'bid'].sort_values('price', ascending=True)
    asks = orderbook_df[orderbook_df['side'] == 'ask'].sort_values('price', ascending=True)
    
    fig.add_trace(go.Scatter(
        x=bids['price'],
        y=bids['cumulative_amount'],
        name='Bids',
        line=dict(color='green'),
        fill='tozeroy'
    ))
    
    # Add asks
    fig.add_trace(go.Scatter(
        x=asks['price'],
        y=asks['cumulative_amount'],
        name='Asks',
        line=dict(color='red'),
        fill='tozeroy'
    ))
    
    fig.update_layout(
        title=f'Order Book Depth - {trading_pair}',
        xaxis_title='Price',
        yaxis_title='Cumulative Amount',
        showlegend=True,
        template='plotly_dark'  # Using dark theme to match your other plots
    )
    
    fig.show()

2025-02-05 23:17:02,717 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16a75d570>
2025-02-05 23:17:02,718 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16a6a6020>, 517654.468689958)])']
connector: <aiohttp.connector.TCPConnector object at 0x16a75d510>



Order Book Data:
    price  amount side  cumulative_amount
0  1.2992      13  bid                 13
1  1.2991     271  bid                284
2   1.299    1626  bid               1910
3  1.2989    4393  bid               6303
4  1.2988    5167  bid              11470
5  1.2987    9471  bid              20941
6  1.2986    4017  bid              24958
7  1.2985    5798  bid              30756
8  1.2984    6519  bid              37275
9  1.2983    8846  bid              46121
10 1.2982    4226  bid              50347
11 1.2981   10171  bid              60518
12  1.298   10596  bid              71114
13 1.2979    8638  bid              79752
14 1.2978    6932  bid              86684
15 1.2977    6622  bid              93306
16 1.2976    3303  bid              96609
17 1.2975    7894  bid             104503
18 1.2974    3303  bid             107806
19 1.2973   17811  bid             125617
20 1.2993    3049  ask               3049
21 1.2994    1601  ask               4650
22 1.2995    165

In [7]:
async def get_orderbook_data():
    connector = None
    try:
        # Get Binance order book through connector
        connector = clob.get_connector("okx_perpetual")
        orderbook = await connector._orderbook_ds._order_book_snapshot(trading_pair)
        
        # Create lists to store the data
        data = []
        
        # Add bids
        for price, amount, _ in orderbook.bids[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'bid'
            })
            
        # Add asks
        for price, amount, _ in orderbook.asks[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'ask'
            })
        
        # Create DataFrame from the list of dictionaries
        orderbook_df = pd.DataFrame(data)
        
        # Calculate cumulative amounts
        orderbook_df['cumulative_amount'] = orderbook_df.groupby('side')['amount'].cumsum()
        
        return orderbook_df
        
    except Exception as e:
        print(f"Error fetching order book data: {str(e)}")
        return None
    finally:
        # Properly close the connector and session
        if connector and hasattr(connector, '_session') and not connector._session.closed:
            await connector._session.close()
        if hasattr(clob, '_session') and not clob._session.closed:
            await clob._session.close()

# Get and display the order book data
orderbook_df = await get_orderbook_data()

if orderbook_df is not None:
    print("\nOrder Book Data:")
    print(orderbook_df)
    
    # You can also create a more detailed view of market depth
    print("\nMarket Depth Analysis:")
    print("\nBids Summary:")
    bids = orderbook_df[orderbook_df['side'] == 'bid'].head()
    print(bids)
    
    print("\nAsks Summary:")
    asks = orderbook_df[orderbook_df['side'] == 'ask'].head()
    print(asks)
    
    # Calculate and print some market metrics
    best_bid = bids['price'].iloc[0]
    best_ask = asks['price'].iloc[0]
    spread = best_ask - best_bid
    spread_pct = (spread / best_bid) * 100
    
    print(f"\nMarket Metrics:")
    print(f"Best Bid: {best_bid:.8f}")
    print(f"Best Ask: {best_ask:.8f}")
    print(f"Spread: {spread:.8f} ({spread_pct:.4f}%)")
    
    # Create a visualization of the order book
    fig = go.Figure()
    
    # Add bids
    bids = orderbook_df[orderbook_df['side'] == 'bid'].sort_values('price', ascending=True)
    asks = orderbook_df[orderbook_df['side'] == 'ask'].sort_values('price', ascending=True)
    
    fig.add_trace(go.Scatter(
        x=bids['price'],
        y=bids['cumulative_amount'],
        name='Bids',
        line=dict(color='green'),
        fill='tozeroy'
    ))
    
    # Add asks
    fig.add_trace(go.Scatter(
        x=asks['price'],
        y=asks['cumulative_amount'],
        name='Asks',
        line=dict(color='red'),
        fill='tozeroy'
    ))
    
    fig.update_layout(
        title=f'Order Book Depth - {trading_pair}',
        xaxis_title='Price',
        yaxis_title='Cumulative Amount',
        showlegend=True,
        template='plotly_dark'  # Using dark theme to match your other plots
    )
    
    fig.show()

2025-02-05 23:17:11,651 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16a76fcd0>
2025-02-05 23:17:11,653 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16a63c580>, 517663.402607708)])']
connector: <aiohttp.connector.TCPConnector object at 0x16a76fc70>



Order Book Data:
    price  amount side  cumulative_amount
0   1.299   48812  bid              48812
1   1.298   79631  bid             128443
2   1.297  132029  bid             260472
3   1.296   54034  bid             314506
4   1.295   69835  bid             384341
5   1.294  117023  bid             501364
6   1.293   20024  bid             521388
7   1.292   47731  bid             569119
8   1.291   10814  bid             579933
9    1.29   41207  bid             621140
10  1.289   55115  bid             676255
11  1.288   50952  bid             727207
12  1.287   45292  bid             772499
13  1.286   82763  bid             855262
14  1.285   25055  bid             880317
15  1.284   66173  bid             946490
16  1.283   15498  bid             961988
17  1.282  159726  bid            1121714
18  1.281    6259  bid            1127973
19   1.28   34066  bid            1162039
20    1.3   13072  ask              13072
21  1.301   50535  ask              63607
22  1.302   8416

In [8]:
async def get_orderbook_data():
    connector = None
    try:
        # Get Binance order book through connector
        connector = clob.get_connector("kucoin")
        orderbook = await connector._orderbook_ds._order_book_snapshot(trading_pair)
        
        # Create lists to store the data
        data = []
        
        # Add bids
        for price, amount, _ in orderbook.bids[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'bid'
            })
            
        # Add asks
        for price, amount, _ in orderbook.asks[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'ask'
            })
        
        # Create DataFrame from the list of dictionaries
        orderbook_df = pd.DataFrame(data)
        
        # Calculate cumulative amounts
        orderbook_df['cumulative_amount'] = orderbook_df.groupby('side')['amount'].cumsum()
        
        return orderbook_df
        
    except Exception as e:
        print(f"Error fetching order book data: {str(e)}")
        return None
    finally:
        # Properly close the connector and session
        if connector and hasattr(connector, '_session') and not connector._session.closed:
            await connector._session.close()
        if hasattr(clob, '_session') and not clob._session.closed:
            await clob._session.close()

# Get and display the order book data
orderbook_df = await get_orderbook_data()

if orderbook_df is not None:
    print("\nOrder Book Data:")
    print(orderbook_df)
    
    # You can also create a more detailed view of market depth
    print("\nMarket Depth Analysis:")
    print("\nBids Summary:")
    bids = orderbook_df[orderbook_df['side'] == 'bid'].head()
    print(bids)
    
    print("\nAsks Summary:")
    asks = orderbook_df[orderbook_df['side'] == 'ask'].head()
    print(asks)
    
    # Calculate and print some market metrics
    best_bid = bids['price'].iloc[0]
    best_ask = asks['price'].iloc[0]
    spread = best_ask - best_bid
    spread_pct = (spread / best_bid) * 100
    
    print(f"\nMarket Metrics:")
    print(f"Best Bid: {best_bid:.8f}")
    print(f"Best Ask: {best_ask:.8f}")
    print(f"Spread: {spread:.8f} ({spread_pct:.4f}%)")
    
    # Create a visualization of the order book
    fig = go.Figure()
    
    # Add bids
    bids = orderbook_df[orderbook_df['side'] == 'bid'].sort_values('price', ascending=True)
    asks = orderbook_df[orderbook_df['side'] == 'ask'].sort_values('price', ascending=True)
    
    fig.add_trace(go.Scatter(
        x=bids['price'],
        y=bids['cumulative_amount'],
        name='Bids',
        line=dict(color='green'),
        fill='tozeroy'
    ))
    
    # Add asks
    fig.add_trace(go.Scatter(
        x=asks['price'],
        y=asks['cumulative_amount'],
        name='Asks',
        line=dict(color='red'),
        fill='tozeroy'
    ))
    
    fig.update_layout(
        title=f'Order Book Depth - {trading_pair}',
        xaxis_title='Price',
        yaxis_title='Cumulative Amount',
        showlegend=True,
        template='plotly_dark'  # Using dark theme to match your other plots
    )
    
    fig.show()

2025-02-05 23:17:14,639 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16a75e7a0>
2025-02-05 23:17:14,643 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16a7eba00>, 517666.38573225)])']
connector: <aiohttp.connector.TCPConnector object at 0x16a75ec50>



Order Book Data:
    price    amount side  cumulative_amount
0  1.2992   51.6327  bid            51.6327
1  1.2991  153.9337  bid           205.5664
2   1.299  192.4377  bid           398.0041
3  1.2989    99.875  bid           497.8791
4  1.2988  454.1359  bid            952.015
5  1.2987 2097.7109  bid          3049.7259
6  1.2986 1659.0527  bid          4708.7786
7  1.2985  693.2721  bid          5402.0507
8  1.2984    613.12  bid          6015.1707
9  1.2982    43.875  bid          6059.0457
10 1.2981  378.6158  bid          6437.6615
11  1.298  171.2715  bid           6608.933
12 1.2979  801.1066  bid          7410.0396
13 1.2977  422.4908  bid          7832.5304
14 1.2976  170.3351  bid          8002.8655
15 1.2975 1970.6077  bid          9973.4732
16 1.2974 6522.8188  bid          16496.292
17 1.2973 7783.2708  bid         24279.5628
18 1.2971  897.0348  bid         25176.5976
19  1.297 4162.4032  bid         29339.0008
20 1.2995    43.875  ask             43.875
21 1.2997  319

In [9]:
async def monitor_large_orders(trading_pair: str, price_threshold_pct: float = 0.01, volume_threshold: float = None):
    connector = None
    try:
        # Get order book through connector
        connector = clob.get_connector("binance_perpetual")
        orderbook = await connector._orderbook_ds._order_book_snapshot(trading_pair)
        
        # Calculate mid price from best bid and ask
        best_bid = float(orderbook.bids[0][0])
        best_ask = float(orderbook.asks[0][0])
        mid_price = (best_bid + best_ask) / 2
        
        price_range = mid_price * price_threshold_pct
        
        # Define price bounds
        lower_bound = mid_price - price_range
        upper_bound = mid_price + price_range
        
        # If volume threshold not set, calculate based on average order size
        if volume_threshold is None:
            all_orders = [float(amount) for _, amount, _ in orderbook.bids[:20] + orderbook.asks[:20]]
            volume_threshold = sum(all_orders) / len(all_orders) * 3  # 3x average order size
        
        large_orders = {
            'bids': [],
            'asks': []
        }
        
        # Check bids
        for price, amount, _ in orderbook.bids[:20]:
            price = float(price)
            amount = float(amount)
            if price >= lower_bound and amount >= volume_threshold:
                large_orders['bids'].append({
                    'price': price,
                    'amount': amount,
                    'distance_from_mid': (mid_price - price) / mid_price * 100
                })
                
        # Check asks
        for price, amount, _ in orderbook.asks[:20]:
            price = float(price)
            amount = float(amount)
            if price <= upper_bound and amount >= volume_threshold:
                large_orders['asks'].append({
                    'price': price,
                    'amount': amount,
                    'distance_from_mid': (price - mid_price) / mid_price * 100
                })
        
        return {
            'mid_price': mid_price,
            'large_orders': large_orders,
            'timestamp': pd.Timestamp.now()
        }
        
    except Exception as e:
        print(f"Error monitoring order book: {str(e)}")
        return None
    finally:
        if connector and hasattr(connector, '_session') and not connector._session.closed:
            await connector._session.close()

async def continuous_monitoring(trading_pair: str, 
                             interval_seconds: int = 5,
                             price_threshold_pct: float = 0.01,
                             volume_threshold: float = None):
    while True:
        result = await monitor_large_orders(
            trading_pair=trading_pair,
            price_threshold_pct=price_threshold_pct,
            volume_threshold=volume_threshold
        )
        
        if result and (result['large_orders']['bids'] or result['large_orders']['asks']):
            print(f"\nLarge orders detected at {result['timestamp']}:")
            print(f"Mid price: {result['mid_price']}")
            
            if result['large_orders']['bids']:
                print("\nLarge bid orders:")
                for order in result['large_orders']['bids']:
                    print(f"Price: {order['price']}, Amount: {order['amount']}, "
                          f"Distance from mid: {order['distance_from_mid']:.2f}%")
            
            if result['large_orders']['asks']:
                print("\nLarge ask orders:")
                for order in result['large_orders']['asks']:
                    print(f"Price: {order['price']}, Amount: {order['amount']}, "
                          f"Distance from mid: {order['distance_from_mid']:.2f}%")
                    
        await asyncio.sleep(interval_seconds)



2025-02-05 23:17:16,722 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16c03aec0>
2025-02-05 23:17:16,724 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16bed38e0>, 517665.224037375)])']
connector: <aiohttp.connector.TCPConnector object at 0x16c03aef0>
2025-02-05 23:17:16,725 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16c0391b0>
2025-02-05 23:17:16,726 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16a7eb580>, 517668.427576291)])']
connector: <aiohttp.connector.TCPConnector object at 0x16c0394e0>


In [10]:
candles.data

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume
timestamp,,,,,,,,,,
2025-01-07 05:00:00,1736226000,2.5689,2.5773,2.5393,2.5611,6824181,17416027.7846,53357,3195747,8157597.2264
2025-01-07 06:00:00,1736229600,2.5611,2.5665,2.5412,2.5483,5742446,14654079.8943,41814,3113780,7945573.1259
2025-01-07 07:00:00,1736233200,2.5483,2.5798,2.5482,2.5758,5025988,12888497.4943,39580,2938178,7536306.3818
2025-01-07 08:00:00,1736236800,2.5759,2.6264,2.5701,2.6256,9963630,25883518.2551,76720,5862477,15228766.9423
2025-01-07 09:00:00,1736240400,2.6256,2.6356,2.5912,2.6103,7275044,18980777.8804,82059,3323780,8673758.2157
...,...,...,...,...,...,...,...,...,...,...
2025-02-06 00:00:00,1738800000,1.287,1.297,1.2749,1.2768,5822248,7492180.8441,41951,2582825,3324572.9322
2025-02-06 01:00:00,1738803600,1.2769,1.2986,1.2681,1.287,6053304,7768592.8097,40464,3066137,3936958.6587
2025-02-06 02:00:00,1738807200,1.287,1.3071,1.2846,1.294,4696024,6086579.48,38136,2362232,3061491.4538


In [11]:
# EMAs
ema_short = 8
ema_medium = 29
ema_long = 31

# MACD
macd_fast = 22
macd_slow = 36
macd_signal = 17

# ATR
atr_length = 3
atr_multiplier = 1.5

# Add indicators
candles_df = candles.data
candles_df.ta.macd(fast=macd_fast, slow=macd_slow, signal=macd_signal, append=True)
candles_df.ta.atr(length=atr_length, append=True)
candles_df.ta.ema(length=ema_short, append=True)
candles_df.ta.ema(length=ema_medium, append=True)
candles_df.ta.ema(length=ema_long, append=True)
candles_df["long_atr_support"] = candles_df["close"].shift(1) - candles_df[f"ATRr_{atr_length}"] * atr_multiplier
candles_df["short_atr_resistance"] = candles_df["close"].shift(1) + candles_df[f"ATRr_{atr_length}"] * atr_multiplier

candles_df.tail(5)

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume,MACD_22_36_17,MACDh_22_36_17,MACDs_22_36_17,ATRr_3,EMA_8,EMA_29,EMA_31,long_atr_support,short_atr_resistance
timestamp,,,,,,,,,,,,,,,,,,,
2025-02-06 00:00:00,1738800000,1.287,1.297,1.2749,1.2768,5822248,7492180.8441,41951,2582825,3324572.9322,-0.0087236,-0.0020614,-0.0066622,0.02608574,1.28860794,1.30938937,1.3105853,1.2479714,1.3262286
2025-02-06 01:00:00,1738803600,1.2769,1.2986,1.2681,1.287,6053304,7768592.8097,40464,3066137,3936958.6587,-0.00884665,-0.00194173,-0.00690492,0.02755716,1.28825062,1.30789674,1.30911122,1.23546426,1.31813574
2025-02-06 02:00:00,1738807200,1.287,1.3071,1.2846,1.294,4696024,6086579.48,38136,2362232,3061491.4538,-0.00868103,-0.00157876,-0.00710226,0.02587144,1.28952826,1.30697029,1.30816677,1.24819284,1.32580716
2025-02-06 03:00:00,1738810800,1.294,1.3064,1.2854,1.2996,3686406,4775862.0078,27447,2013398,2609354.6806,-0.00831292,-0.00107614,-0.00723678,0.02424763,1.29176643,1.30647894,1.30763135,1.25762856,1.33037144
2025-02-06 04:00:00,1738814400,1.2996,1.3038,1.2956,1.3002,918087,1192516.8933,7774,505147,656141.4055,-0.00793617,-0.00062168,-0.00731449,0.01889842,1.29364055,1.30606035,1.30716689,1.27125237,1.32794763


In [12]:
async def generate_trading_signal(trading_pair: str, candles_df: pd.DataFrame, orderbook_data: dict):
    try:
        # Technical Analysis Signals
        macdh = candles_df[f"MACDh_{macd_fast}_{macd_slow}_{macd_signal}"]
        short_ema = candles_df[f"EMA_{ema_short}"]
        medium_ema = candles_df[f"EMA_{ema_medium}"]
        long_ema = candles_df[f"EMA_{ema_long}"]
        close = candles_df["close"]
        
        # Order Book Signals
        mid_price = orderbook_data['mid_price']
        large_bids = orderbook_data['large_orders']['bids']
        large_asks = orderbook_data['large_orders']['asks']
        
        # Calculate order book pressure
        bid_pressure = sum(order['amount'] for order in large_bids)
        ask_pressure = sum(order['amount'] for order in large_asks)
        ob_imbalance = (bid_pressure - ask_pressure) / (bid_pressure + ask_pressure) if (bid_pressure + ask_pressure) > 0 else 0
        
        # Combined Signals
        long_signal = (
            (short_ema.iloc[-1] > medium_ema.iloc[-1]) and 
            (medium_ema.iloc[-1] > long_ema.iloc[-1]) and 
            (close.iloc[-1] > short_ema.iloc[-1]) and 
            (macdh.iloc[-1] > 0) and
            (ob_imbalance > 0.2)  # Significant buy pressure
        )
        
        short_signal = (
            (short_ema.iloc[-1] < medium_ema.iloc[-1]) and 
            (medium_ema.iloc[-1] < long_ema.iloc[-1]) and 
            (close.iloc[-1] < short_ema.iloc[-1]) and 
            (macdh.iloc[-1] < 0) and
            (ob_imbalance < -0.2)  # Significant sell pressure
        )
        
        # Generate signal
        if long_signal:
            return {
                'signal': 'LONG',
                'entry_price': mid_price,
                'stop_loss': mid_price * 0.99,  # 1% stop loss
                'take_profit': mid_price * 1.02,  # 2% take profit
                'timestamp': pd.Timestamp.now(),
                'metrics': {
                    'ob_imbalance': ob_imbalance,
                    'macd_histogram': macdh.iloc[-1],
                    'ema_alignment': 'bullish'
                }
            }
        elif short_signal:
            return {
                'signal': 'SHORT',
                'entry_price': mid_price,
                'stop_loss': mid_price * 1.01,  # 1% stop loss
                'take_profit': mid_price * 0.98,  # 2% take profit
                'timestamp': pd.Timestamp.now(),
                'metrics': {
                    'ob_imbalance': ob_imbalance,
                    'macd_histogram': macdh.iloc[-1],
                    'ema_alignment': 'bearish'
                }
            }
        
        return {
            'signal': 'NEUTRAL',
            'timestamp': pd.Timestamp.now(),
            'metrics': {
                'ob_imbalance': ob_imbalance,
                'macd_histogram': macdh.iloc[-1],
                'ema_alignment': 'neutral'
            }
        }
        
    except Exception as e:
        print(f"Error generating trading signal: {str(e)}")
        return None

async def continuous_strategy_monitoring(
    trading_pair: str,
    interval_seconds: int = 5,
    price_threshold_pct: float = 0.01,
    volume_threshold: float = 1000
):
    while True:
        try:
            # Get order book data
            ob_data = await monitor_large_orders(
                trading_pair=trading_pair,
                price_threshold_pct=price_threshold_pct,
                volume_threshold=volume_threshold
            )
            
            # Get latest candle data
            candles = await clob.get_candles_last_days(
                "binance_perpetual", 
                trading_pair, 
                interval="1h",  # Changed from timeframe to interval
                days=1
            )
            
            # Generate trading signal
            if ob_data and candles is not None:
                signal = await generate_trading_signal(
                    trading_pair=trading_pair,
                    candles_df=candles.data,
                    orderbook_data=ob_data
                )
                
                if signal and signal['signal'] != 'NEUTRAL':
                    print(f"\nTrading Signal Generated at {signal['timestamp']}:")
                    print(f"Signal: {signal['signal']}")
                    print(f"Entry Price: {signal['entry_price']}")
                    if 'stop_loss' in signal:
                        print(f"Stop Loss: {signal['stop_loss']}")
                        print(f"Take Profit: {signal['take_profit']}")
                    print("\nMetrics:")
                    for key, value in signal['metrics'].items():
                        print(f"{key}: {value}")
                    
        except Exception as e:
            print(f"Error in strategy monitoring: {str(e)}")
            
        await asyncio.sleep(interval_seconds)

# Start the strategy with the same parameters
await continuous_strategy_monitoring(
    trading_pair=trading_pair,
    interval_seconds=5,
    price_threshold_pct=0.01,
    volume_threshold=1000
)

2025-02-05 23:17:19,184 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16c0b3250>
2025-02-05 23:17:19,185 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16a7eb580>, 517670.936756583)])']
connector: <aiohttp.connector.TCPConnector object at 0x16c0b31f0>
2025-02-05 23:17:22,378 - hummingbot.data_feed.candles_feed.binance_perpetual_candles.binance_perpetual_candles - ERROR - Error fetching historical candles: 'timestamp'
Traceback (most recent call last):
  File "/opt/anaconda3/envs/quants-lab/lib/python3.10/site-packages/hummingbot/data_feed/candles_feed/candles_base.py", line 185, in get_historical_candles
    (candles_df["timestamp"] <= config.end_time) & (candles_df["timestamp"] >= config.start_time)]
  File "/opt/anaconda3/envs/quants-lab/lib/python3.10/site-packages/pandas/core/frame.py", line 4102, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/op

Error in strategy monitoring: 'timestamp'


2025-02-05 23:17:28,745 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16c038c10>
2025-02-05 23:17:28,747 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16a4ce020>, 517680.497618333)])']
connector: <aiohttp.connector.TCPConnector object at 0x16c0389d0>
2025-02-05 23:17:32,180 - hummingbot.data_feed.candles_feed.binance_perpetual_candles.binance_perpetual_candles - ERROR - Error fetching historical candles: 'timestamp'
Traceback (most recent call last):
  File "/opt/anaconda3/envs/quants-lab/lib/python3.10/site-packages/hummingbot/data_feed/candles_feed/candles_base.py", line 185, in get_historical_candles
    (candles_df["timestamp"] <= config.end_time) & (candles_df["timestamp"] >= config.start_time)]
  File "/opt/anaconda3/envs/quants-lab/lib/python3.10/site-packages/pandas/core/frame.py", line 4102, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/op

Error in strategy monitoring: 'timestamp'


2025-02-05 23:17:39,554 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16c3649d0>
2025-02-05 23:17:39,555 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16c35cc40>, 517691.306731875)])']
connector: <aiohttp.connector.TCPConnector object at 0x16c364a00>
2025-02-05 23:17:42,727 - hummingbot.data_feed.candles_feed.binance_perpetual_candles.binance_perpetual_candles - ERROR - Error fetching historical candles: 'timestamp'
Traceback (most recent call last):
  File "/opt/anaconda3/envs/quants-lab/lib/python3.10/site-packages/hummingbot/data_feed/candles_feed/candles_base.py", line 185, in get_historical_candles
    (candles_df["timestamp"] <= config.end_time) & (candles_df["timestamp"] >= config.start_time)]
  File "/opt/anaconda3/envs/quants-lab/lib/python3.10/site-packages/pandas/core/frame.py", line 4102, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/op

Error in strategy monitoring: 'timestamp'


2025-02-05 23:17:44,731 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16a75db10>
2025-02-05 23:17:44,731 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16c0b11b0>
2025-02-05 23:17:44,732 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16a76cdc0>
2025-02-05 23:17:44,732 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x103d22740>, 517682.600816875)])']
connector: <aiohttp.connector.TCPConnector object at 0x16c03b0d0>
2025-02-05 23:17:44,733 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16c364460>
2025-02-05 23:17:44,734 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16c35d420>, 517692.877315166)])']
connector: <aiohttp.connector.TCPConnector obj

CancelledError: 

In [ ]:
async def generate_trading_signal(trading_pair: str, candles_df: pd.DataFrame, orderbook_data: dict):
    try:
        # Technical Analysis Signals (from your existing code)
        macdh = candles_df[f"MACDh_{macd_fast}_{macd_slow}_{macd_signal}"]
        short_ema = candles_df[f"EMA_{ema_short}"]
        medium_ema = candles_df[f"EMA_{ema_medium}"]
        long_ema = candles_df[f"EMA_{ema_long}"]
        close = candles_df["close"]
        
        # Order Book Signals
        mid_price = orderbook_data['mid_price']
        large_bids = orderbook_data['large_orders']['bids']
        large_asks = orderbook_data['large_orders']['asks']
        
        # Calculate order book pressure
        bid_pressure = sum(order['amount'] for order in large_bids)
        ask_pressure = sum(order['amount'] for order in large_asks)
        ob_imbalance = (bid_pressure - ask_pressure) / (bid_pressure + ask_pressure) if (bid_pressure + ask_pressure) > 0 else 0
        
        # Combined Signals
        long_signal = (
            (short_ema.iloc[-1] > medium_ema.iloc[-1]) and 
            (medium_ema.iloc[-1] > long_ema.iloc[-1]) and 
            (close.iloc[-1] > short_ema.iloc[-1]) and 
            (close.iloc[-1] > candles_df["long_atr_support"].iloc[-1]) and 
            (macdh.iloc[-1] > 0) and
            (ob_imbalance > 0.2)  # Significant buy pressure
        )
        
        short_signal = (
            (short_ema.iloc[-1] < medium_ema.iloc[-1]) and 
            (medium_ema.iloc[-1] < long_ema.iloc[-1]) and 
            (close.iloc[-1] < short_ema.iloc[-1]) and 
            (close.iloc[-1] < candles_df["short_atr_resistance"].iloc[-1]) and 
            (macdh.iloc[-1] < 0) and
            (ob_imbalance < -0.2)  # Significant sell pressure
        )
        
        # Generate signal
        if long_signal:
            return {
                'signal': 'LONG',
                'entry_price': mid_price,
                'stop_loss': candles_df["long_atr_support"].iloc[-1],
                'take_profit': mid_price * 1.02,  # 2% take profit
                'timestamp': pd.Timestamp.now(),
                'metrics': {
                    'ob_imbalance': ob_imbalance,
                    'macd_histogram': macdh.iloc[-1],
                    'ema_alignment': 'bullish'
                }
            }
        elif short_signal:
            return {
                'signal': 'SHORT',
                'entry_price': mid_price,
                'stop_loss': candles_df["short_atr_resistance"].iloc[-1],
                'take_profit': mid_price * 0.98,  # 2% take profit
                'timestamp': pd.Timestamp.now(),
                'metrics': {
                    'ob_imbalance': ob_imbalance,
                    'macd_histogram': macdh.iloc[-1],
                    'ema_alignment': 'bearish'
                }
            }
        
        return {
            'signal': 'NEUTRAL',
            'timestamp': pd.Timestamp.now(),
            'metrics': {
                'ob_imbalance': ob_imbalance,
                'macd_histogram': macdh.iloc[-1],
                'ema_alignment': 'neutral'
            }
        }
        
    except Exception as e:
        print(f"Error generating trading signal: {str(e)}")
        return None

In [10]:
async def continuous_monitoring(trading_pair: str, 
                             interval_seconds: int = 5,
                             price_threshold_pct: float = 0.01,
                             volume_threshold: float = None):
    while True:
        result = await monitor_large_orders(
            trading_pair=trading_pair,
            price_threshold_pct=price_threshold_pct,
            volume_threshold=volume_threshold
        )
        
        if result and (result['large_orders']['bids'] or result['large_orders']['asks']):
            print(f"\nLarge orders detected at {result['timestamp']}:")
            print(f"Last price: {result['last_price']}")
            
            if result['large_orders']['bids']:
                print("\nLarge bid orders:")
                for order in result['large_orders']['bids']:
                    print(f"Price: {order['price']}, Amount: {order['amount']}, "
                          f"Distance from last: {order['distance_from_mid']:.2f}%")
            
            if result['large_orders']['asks']:
                print("\nLarge ask orders:")
                for order in result['large_orders']['asks']:
                    print(f"Price: {order['price']}, Amount: {order['amount']}, "
                          f"Distance from last: {order['distance_from_mid']:.2f}%")
                    
        await asyncio.sleep(interval_seconds)

In [ ]:
candles.data

In [ ]:
candles.plot(type="returns", height=400, width=800)

In [ ]:
# EMAs
ema_short = 8
ema_medium = 29
ema_long = 31

# MACD
macd_fast = 22
macd_slow = 36
macd_signal = 17

# ATR
atr_length = 3
atr_multiplier = 1.5

# Add indicators
candles_df = candles.data
candles_df.ta.macd(fast=macd_fast, slow=macd_slow, signal=macd_signal, append=True)
candles_df.ta.atr(length=atr_length, append=True)
candles_df.ta.ema(length=ema_short, append=True)
candles_df.ta.ema(length=ema_medium, append=True)
candles_df.ta.ema(length=ema_long, append=True)
candles_df["long_atr_support"] = candles_df["close"].shift(1) - candles_df[f"ATRr_{atr_length}"] * atr_multiplier
candles_df["short_atr_resistance"] = candles_df["close"].shift(1) + candles_df[f"ATRr_{atr_length}"] * atr_multiplier

candles_df.tail(5)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, 
                    subplot_titles=(trading_pair, 'MACD'),
                    row_heights=[0.7, 0.3])

# Add candlestick
fig.add_trace(go.Candlestick(x=candles_df.index,
                             open=candles_df['open'],
                             high=candles_df['high'],
                             low=candles_df['low'],
                             close=candles_df['close'],
                             name='OHLC'),
              row=1, col=1)

# Add EMAs
ema_fast = f'EMA_{ema_short}'
ema_med = f'EMA_{ema_medium}' 
ema_slow = f'EMA_{ema_long}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_fast],
                         line=dict(color='#00FF00', width=2),
                         name='Fast EMA'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_med],
                         line=dict(color='#FFA500', width=2), 
                         name='Medium EMA'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow],
                         line=dict(color='#0000FF', width=2),
                         name='Slow EMA'), row=1, col=1)

# Add MACD
macd = f'MACD_{macd_fast}_{macd_slow}_{macd_signal}'
macd_s = f'MACDs_{macd_fast}_{macd_slow}_{macd_signal}'
macd_hist = f'MACDh_{macd_fast}_{macd_slow}_{macd_signal}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[macd], 
                         line=dict(color='#00FFFF', width=2),
                         name='MACD'), row=2, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[macd_s], 
                         line=dict(color='#FFA500', width=2),
                         name='Signal'), row=2, col=1)
fig.add_trace(go.Bar(x=candles_df.index, y=candles_df[macd_hist], name='Histogram',
                     marker_color=candles_df[macd_hist].apply(
                         lambda x: '#00FF00' if x >= 0 else '#FF0000')),
                    row=2, col=1)

# Add ATR support and resistance
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df["long_atr_support"],
                         line=dict(color='#00FF00', width=2),
                         name='Long ATR Support'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df["short_atr_resistance"],
                         line=dict(color='#FF0000', width=2),
                         name='Short ATR Resistance'), row=1, col=1)

# Update layout for dark theme
fig.update_layout(
    title=f'{exchange} - {trading_pair} - {timeframe}',
    width=1200, height=800,
    font=dict(color='#e1e1e1'),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    xaxis_rangeslider_visible=False,
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='MACD', showgrid=False),
    showlegend=False
)

# Update axes
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# Show the plot
fig.show()

In [14]:
# Generate signal


macdh = candles_df[f"MACDh_{macd_fast}_{macd_slow}_{macd_signal}"]
short_ema = candles_df[f"EMA_{ema_short}"]
medium_ema = candles_df[f"EMA_{ema_medium}"]
long_ema = candles_df[f"EMA_{ema_long}"]
close = candles_df["close"]


long_condition = (short_ema > medium_ema) & (medium_ema > long_ema) & (close > short_ema) & (close > candles_df["long_atr_support"]) & (macdh > 0) 
short_condition = (short_ema < medium_ema) & (medium_ema < long_ema) & (close < short_ema) & (close < candles_df["short_atr_resistance"]) & (macdh < 0)

candles_df["signal"] = 0
candles_df.loc[long_condition, "signal"] = 1
candles_df.loc[short_condition, "signal"] = -1

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.02,
                    subplot_titles=('OHLC with BB', 'MACD', 'Signal'),
                    row_heights=[0.6, 0.2, 0.2])

# Add candlestick
fig.add_trace(go.Candlestick(x=candles_df.index,
                             open=candles_df['open'],
                             high=candles_df['high'],
                             low=candles_df['low'],
                             close=candles_df['close'],
                             name='Candlesticks'),
              row=1, col=1)


# Add EMAs
ema_fast = f'EMA_{ema_short}'
ema_med = f'EMA_{ema_medium}' 
ema_slow = f'EMA_{ema_long}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_fast],
                         line=dict(color='#00FF00', width=2),
                         name='Fast EMA'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_med],
                         line=dict(color='#FFA500', width=2), 
                         name='Medium EMA'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow],
                         line=dict(color='#0000FF', width=2),
                         name='Slow EMA'), row=1, col=1)

# Add MACD
macd = f'MACD_{macd_fast}_{macd_slow}_{macd_signal}'
macd_s = f'MACDs_{macd_fast}_{macd_slow}_{macd_signal}'
macd_hist = f'MACDh_{macd_fast}_{macd_slow}_{macd_signal}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[macd], 
                         line=dict(color='#00FFFF', width=2),
                         name='MACD'), row=2, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[macd_s], 
                         line=dict(color='#FFA500', width=2),
                         name='Signal'), row=2, col=1)
fig.add_trace(go.Bar(x=candles_df.index, y=candles_df[macd_hist], name='Histogram',
                     marker_color=candles_df[macd_hist].apply(
                         lambda x: '#00FF00' if x >= 0 else '#FF0000')),
              row=2, col=1)

# Add ATR support and resistance
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df["long_atr_support"],
                         line=dict(color='#00FF00', width=2),
                         name='Long ATR Support'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df["short_atr_resistance"],
                         line=dict(color='#FF0000', width=2),
                         name='Short ATR Resistance'), row=1, col=1)

# Add the signal line
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df['signal'],
                         mode='lines',
                         name='Signal',
                         line=dict(color="white")),
              row=3, col=1)

# Update layout for dark theme
fig.update_layout(
    title=f'{exchange} - {trading_pair} - {timeframe}',
    width=1500, height=1000,
    font=dict(color='#e1e1e1'),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    xaxis_rangeslider_visible=False,
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='MACD', showgrid=False),
    yaxis3=dict(title='Signal', showgrid=False),
    showlegend=False
)

# Update axes
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# Show the plot
fig.show()


# CONCLUSION

In this notebook, we have implemented a strategy combining the MACD (Moving Average Convergence Divergence) indicator with Bollinger Bands. We've visualized these indicators along with the price data and generated signals based on their interactions. This approach provides a solid foundation for our trading strategy.
 
## Key components of our strategy include:
 1. MACD for trend identification
 2. Bollinger Bands for volatility measurement and potential reversal points
 3. A signal line derived from the combination of these indicators
 
 The next step is to backtest this strategy to evaluate its profitability and robustness. For this purpose, we have created a controller file named `macd_bb.py` in this folder. This file implements the logic we've developed here, allowing us to conduct comprehensive backtests in the subsequent notebook.